<a href="https://colab.research.google.com/github/BBWorksB/LLM_MaterClass/blob/main/01_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. Train with Scikit-Learn

Moringa Masterclass: Machine Learning End to End

This is part 1 of 3. In this notebook we load a customer churn dataset, do a quick exploratory look, build a Scikit-Learn pipeline, train it, and evaluate it.

Runs fully in Google Colab. No local setup required.

In [1]:
# If running in Colab, this installs the packages we need.
# If running locally with the repo's requirements already installed, this is a no-op.
%pip install -q scikit-learn pandas numpy matplotlib

## 1. Load the data

We are using a customer churn dataset from a telecom company: one row per customer, with account details (contract type, monthly charges, tenure, services subscribed) and a label, `Churn`, that says whether the customer left.

The CSV is bundled in this repo's `data/` folder, so there is no separate download step.

In [3]:
!git clone https://github.com/BBWorksB/LLM_MaterClass
!ls

Cloning into 'LLM_MaterClass'...
fatal: could not read Username for 'https://github.com': No such device or address
sample_data


In [2]:
import pandas as pd

df = pd.read_csv("data/Telco-Customer-Churn.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/Telco-Customer-Churn.csv'

## 2. Quick look at the data

A few things worth checking before training anything: class balance, missing values, and a couple of features that plausibly relate to churn.

In [ ]:
df["Churn"].value_counts(normalize=True)

In [ ]:
# TotalCharges is sometimes read as text because of blank strings for brand-new customers.
# Coerce to numeric and check for anything that failed to convert.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.isna().sum().sum()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.groupby("Churn")["tenure"].mean().plot(kind="bar", ax=axes[0], title="Avg tenure by churn")
df.groupby("Churn")["MonthlyCharges"].mean().plot(kind="bar", ax=axes[1], title="Avg monthly charges by churn")
plt.tight_layout()
plt.show()

Two quick reads: customers who churn tend to have shorter tenure and higher monthly charges. That is the kind of signal we want a model to pick up on, and it is also useful intuition to carry into the explainability part later.

## 3. Build a Scikit-Learn pipeline

We separate numeric and categorical columns, scale the numeric ones, one-hot encode the categorical ones, and feed the result into a classifier. Wrapping this in a single `Pipeline` means the exact same preprocessing is applied at training time and at prediction time, which matters once this leaves the notebook.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

target = "Churn"
drop_cols = ["customerID", target]

X = df.drop(columns=drop_cols)
y = (df[target] == "Yes").astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

log_reg_pipeline.fit(X_train, y_train)

## 4. Evaluate

Accuracy alone is misleading here because churn is imbalanced (roughly 1 in 4 customers churns in this sample). We look at precision, recall, F1, and ROC-AUC as well.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["No churn", "Churn"])
plt.show()

## 5. A second model, for comparison

Let's also try a Random Forest. Same features, same split, different algorithm. This gives us two trained models to compare in the next notebook, when we bring in MLflow.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=["No churn", "Churn"]))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_rf), 3))

## Recap

We now have two trained pipelines (Logistic Regression and Random Forest) and a held-out test set to compare them on. Right now, if you closed this notebook, that comparison would be gone. That is the gap the next notebook fixes.

Next: `02_track_mlflow.ipynb`, where we log both models' parameters, metrics, and artifacts with MLflow, and register the better one.